# Notebook 10 — OpenAI Predictive Model: Document Quality, Embeddings & Prompt Version → Extraction Outcomes

**Objective:** Model the probability of each extraction outcome as a function of:

1. **Document quality** — OCR blur score, contrast, sharpness (Laplacian, Tenengrad, RMS contrast)
2. **Embedding features** — `text-embedding-3-large` dense embeddings of extracted case text (3072-dim → PCA-reduced)
3. **Safety / text complexity metrics** — negation rate, uncertainty language rate, lexical diversity, token count
4. **Prompt method / version** — one-hot encoded (RLS, BFOP, 2POP, zero_shot, chain_of_thought, rag, few_shot, …)

**Three binary outcome models:**
- **Model A**: P(correct extraction) per observation
- **Model B**: P(omission) per observation  
- **Model C**: P(fabrication/hallucination) per observation

**Pipeline:**
```
extracted_text/*.txt
    → text-embedding-3-large API → 3072-dim embeddings
    → PCA(50) reduction
    ↘
OCR quality CSV (NB08)         ↘
Text features CSV (NB05)    → concatenate → train/test split
Prompt version (one-hot)       ↗
    → Logistic Regression + XGBoost → Three binary models
    → Ridge Regression (element-level) → Outcome rates
    → SHAP feature importance per outcome
```

**Outputs:**
- `data/processed/openai_embeddings.csv` — case-level 3072-dim embeddings (cached)
- `data/processed/openai_pca_features.csv` — PCA-reduced embedding features
- `reports/binary_model_results.csv` — per-model accuracy, AUC, calibration
- `reports/feature_importance_by_outcome.png` — SHAP plots for each model
- `reports/calibration_curves.png` — probability calibration for each outcome

In [ ]:
import os
import warnings
import json
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm

from openai import OpenAI

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, brier_score_loss, accuracy_score,
    classification_report, mean_squared_error, r2_score,
    CalibrationDisplay
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
load_dotenv()

PROJECT_ROOT     = Path(os.getenv('PROJECT_ROOT',
    r'C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca'))
DATA_PRIVATE_DIR = Path(os.getenv('DATA_PRIVATE_DIR', r'C:\Users\jamesr4\loc\data_private'))

TEXT_DIR   = DATA_PRIVATE_DIR / 'extracted_text'
PROC_DIR   = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODEL  = 'text-embedding-3-large'
EMBED_DIM    = 3072
PCA_COMPONENTS = 50
CV_FOLDS     = 5
RANDOM_STATE = 42

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print(f'Embedding model : {EMBED_MODEL} → PCA({PCA_COMPONENTS})')
print(f'TEXT_DIR exists : {TEXT_DIR.exists()}')

## 10.1 — Generate / Load OpenAI Embeddings (cached)

In [ ]:
EMBED_CACHE = PROC_DIR / 'openai_embeddings.csv'

def get_embedding(text: str, model: str = EMBED_MODEL) -> list:
    """Call OpenAI Embeddings API and return vector."""
    text = text.replace('\n', ' ').strip()
    if not text:
        return [0.0] * EMBED_DIM
    resp = client.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding

# Load or generate embeddings
if EMBED_CACHE.exists():
    print(f'Loading cached embeddings from {EMBED_CACHE}')
    df_embed = pd.read_csv(EMBED_CACHE, index_col='case_id')
    print(f'  Shape: {df_embed.shape}')
else:
    if not TEXT_DIR.exists():
        raise FileNotFoundError(
            f'TEXT_DIR not found: {TEXT_DIR}\n'
            'Run NB04 (source doc text extraction) first.')

    txt_files = sorted(TEXT_DIR.glob('CASE_*.txt'))
    print(f'Generating embeddings for {len(txt_files)} cases via {EMBED_MODEL}...')
    print('Estimated cost: ~$0.00013 per 1K tokens × avg ~500 tokens/doc')

    rows = {}
    for txt_path in tqdm(txt_files, desc='Embedding cases'):
        case_id = txt_path.stem
        text = txt_path.read_text(encoding='utf-8', errors='ignore')
        # Truncate to ~8000 chars (~2000 tokens) to keep cost low
        text = text[:8000]
        try:
            vec = get_embedding(text)
            rows[case_id] = vec
        except Exception as exc:
            print(f'  SKIP {case_id}: {exc}')

    embed_cols = [f'emb_{i}' for i in range(EMBED_DIM)]
    df_embed = pd.DataFrame.from_dict(rows, orient='index', columns=embed_cols)
    df_embed.index.name = 'case_id'
    df_embed.to_csv(EMBED_CACHE)
    print(f'Saved embeddings: {EMBED_CACHE}  shape={df_embed.shape}')

## 10.2 — PCA-Reduce Embeddings

In [ ]:
PCA_CACHE = PROC_DIR / 'openai_pca_features.csv'

if PCA_CACHE.exists():
    df_pca = pd.read_csv(PCA_CACHE, index_col='case_id')
    print(f'Loaded PCA features: {df_pca.shape}')
else:
    scaler = StandardScaler()
    pca    = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)
    X_scaled   = scaler.fit_transform(df_embed.values)
    X_pca      = pca.fit_transform(X_scaled)
    var_explained = pca.explained_variance_ratio_.cumsum()[-1]
    print(f'PCA({PCA_COMPONENTS}) explains {var_explained:.1%} of variance')

    pca_cols = [f'pca_{i}' for i in range(PCA_COMPONENTS)]
    df_pca   = pd.DataFrame(X_pca, index=df_embed.index, columns=pca_cols)
    df_pca.index.name = 'case_id'
    df_pca.to_csv(PCA_CACHE)
    print(f'Saved PCA features: {PCA_CACHE}')

# Scree plot
if 'pca' in dir():
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.cumsum(pca.explained_variance_ratio_) * 100,
            marker='o', markersize=4, linewidth=1.5, color='#2980b9')
    ax.axhline(80, color='red', linestyle='--', linewidth=1, label='80% threshold')
    ax.set_xlabel('Number of PCA components')
    ax.set_ylabel('Cumulative variance explained (%)')
    ax.set_title(f'PCA Scree Plot — {EMBED_MODEL} embeddings', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'embedding_pca_scree.png', dpi=200, bbox_inches='tight')
    plt.show()

## 10.3 — Assemble Feature Matrix

Joins:
1. PCA-reduced embeddings (from NB10 §10.2)
2. OCR quality features (from NB08 / NB05 outputs)
3. Text complexity features (from NB05)
4. Prompt method one-hot (from ground-truth dataset)

In [ ]:
# ── Load available feature files ───────────────────────────────────────────────
feature_dfs = [df_pca.reset_index()]   # always have embeddings

# OCR quality (NB08 blur scan results)
blur_path = REPORT_DIR / 'ocr_blur_scan_results.csv'
if blur_path.exists():
    df_blur = pd.read_csv(blur_path)
    # Aggregate to case level: mean blur_score, pct_blurry
    # Extract case_id from relative_path (filename stem)
    df_blur['case_id'] = df_blur['relative_path'].apply(
        lambda p: Path(p).stem)
    df_blur_case = df_blur.groupby('case_id').agg(
        blur_score_mean=('blur_score', 'mean'),
        blur_score_min=('blur_score', 'min'),
        pct_blurry=('is_blurry', 'mean'),
        n_pages=('page', 'count')
    ).reset_index()
    feature_dfs.append(df_blur_case)
    print(f'OCR quality features: {df_blur_case.shape}')
else:
    print('OCR blur results not found — run NB08 first. Skipping.')

# Case-level OCR quality (NB05)
ocr_qual_path = PROC_DIR.parent / 'features' / 'case_level_ocr_quality.csv'
if not ocr_qual_path.exists():
    ocr_qual_path = PROC_DIR / 'case_level_ocr_quality.csv'
if ocr_qual_path.exists():
    df_ocr = pd.read_csv(ocr_qual_path)
    feature_dfs.append(df_ocr)
    print(f'NB05 OCR quality loaded: {df_ocr.shape}')

# Text complexity features (NB05)
txt_feat_path = PROC_DIR.parent / 'features' / 'case_text_features.csv'
if not txt_feat_path.exists():
    txt_feat_path = PROC_DIR / 'case_text_features.csv'
if txt_feat_path.exists():
    df_text_feat = pd.read_csv(txt_feat_path)
    feature_dfs.append(df_text_feat)
    print(f'Text features loaded: {df_text_feat.shape}')

# Merge all feature frames on case_id
df_features = feature_dfs[0]
for fdf in feature_dfs[1:]:
    if 'case_id' in fdf.columns:
        df_features = df_features.merge(fdf, on='case_id', how='left')

print(f'\nFinal feature matrix: {df_features.shape}')
print(f'Feature columns: {[c for c in df_features.columns if c != "case_id"][:10]} ...')

## 10.4 — Build Outcome Variables

Creating three binary outcomes: is_correct, is_omission, is_fabrication

In [ ]:
# Load comprehensive dataset (observation-level)
comp_path = PROC_DIR / 'comprehensive_enhanced_dataset_with_all_metrics.csv'
df_comp = pd.read_csv(comp_path, index_col=0)
print(f'Comprehensive dataset: {df_comp.shape}')

# ── Create three binary outcomes from AI status columns ───────────────────────
# AI status columns: *_ai_status (1=correct, 2=omission, 3=fabrication)
ai_status_cols = [c for c in df_comp.columns if c.endswith('_ai_status')]
print(f'AI status columns found: {len(ai_status_cols)}')

# Stack to observation-level: one row per (case × element)
obs_rows = []
for col in ai_status_cols:
    element = col.replace('_ai_status', '')
    for idx, row in df_comp.iterrows():
        val = row[col]
        if pd.isna(val):
            continue
        # Create three binary outcomes
        status = int(val)
        obs_rows.append({
            'obs_id': idx,
            'element': element,
            'is_correct': 1 if status == 1 else 0,
            'is_omission': 1 if status == 2 else 0,
            'is_fabrication': 1 if status == 3 else 0,
            'tumor_type': row.get('tumor_invasive_dcis', np.nan),
            'complex': row.get('complex_case_status', np.nan),
        })
df_obs = pd.DataFrame(obs_rows)
print(f'Observation-level rows: {len(df_obs)}')

# Outcome distributions
for outcome in ['is_correct', 'is_omission', 'is_fabrication']:
    count = df_obs[outcome].sum()
    pct = count / len(df_obs) * 100
    label = outcome.replace('is_', '').title()
    print(f'  {label}: {count} ({pct:.1f}%)')

# Check mutual exclusivity
print(f'\nMutual exclusivity check:')
print(f'  Cases with multiple outcomes: {(df_obs[["is_correct", "is_omission", "is_fabrication"]].sum(axis=1) > 1).sum()}')
print(f'  Cases with no outcome: {(df_obs[["is_correct", "is_omission", "is_fabrication"]].sum(axis=1) == 0).sum()}')

# ── Outcome D: per-element (run-level) outcome rates ───────────────────────────
elem_status_cols = [(c.replace('_ai_status',''), c) for c in ai_status_cols]
run_rows = []
for element, col in elem_status_cols:
    valid = df_comp[col].dropna()
    run_rows.append({
        'element': element,
        'n_cases': len(valid),
        'accuracy': (valid == 1).mean(),
        'omission_rate': (valid == 2).mean(),
        'fabrication_rate': (valid == 3).mean(),
    })
df_run = pd.DataFrame(run_rows)

print(f'\nRun-level rows (per element): {len(df_run)}')
display(df_run.sort_values('accuracy').round(3))

## 10.5 — Three Binary Models for Outcome Prediction

Features: PCA embeddings + OCR quality + text complexity  
Targets: is_correct, is_omission, is_fabrication (three separate binary models)  
Models: Logistic Regression + XGBoost for each outcome (5-fold CV)

In [ ]:
# Build design matrix for binary models
# Numeric feature columns only (exclude case_id, identifiers)
feat_cols = [c for c in df_features.columns
             if c != 'case_id' and df_features[c].dtype in [np.float64, np.int64, float, int]]
print(f'Feature columns available: {len(feat_cols)}')

# Element one-hot encoding
from sklearn.preprocessing import LabelEncoder
le_elem = LabelEncoder()
df_obs['element_enc'] = le_elem.fit_transform(df_obs['element'])

# Build X matrix (same for all three models)
X_base = pd.get_dummies(df_obs[['element', 'tumor_type', 'complex']], drop_first=True)

# If case-level features are available and indexable, join them
if len(df_features) > 0 and len(feat_cols) > 0:
    # Use obs_id as row index into df_features (up to available rows)
    n_feat_rows = len(df_features)
    feat_indices = df_obs['obs_id'].values % n_feat_rows  # modulo fallback
    X_feats = df_features[feat_cols].fillna(df_features[feat_cols].median()).values[feat_indices]
    X = np.hstack([X_base.fillna(0).values, X_feats])
    print(f'X shape (with case features): {X.shape}')
else:
    X = X_base.fillna(0).values
    print(f'X shape (element/covariate features only): {X.shape}')

# Define three target variables
y_correct = df_obs['is_correct'].values
y_omission = df_obs['is_omission'].values
y_fabrication = df_obs['is_fabrication'].values

print(f'\nTarget distributions:')
print(f'  Correct: {y_correct.mean():.3f} ({y_correct.sum()} cases)')
print(f'  Omission: {y_omission.mean():.3f} ({y_omission.sum()} cases)')
print(f'  Fabrication: {y_fabrication.mean():.3f} ({y_fabrication.sum()} cases)')

In [ ]:
# Train three binary models
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Define models
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, C=0.1, 
                                   class_weight='balanced',
                                   random_state=RANDOM_STATE))
])
pipe_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                          learning_rate=0.05, random_state=RANDOM_STATE))
])

# Store results for all three outcomes
outcomes = [
    ('Correct', y_correct),
    ('Omission', y_omission),
    ('Fabrication', y_fabrication)
]

all_results = []
all_probs = {}
best_models = {}

for outcome_name, y in outcomes:
    print(f'\n=== Model for {outcome_name} ===')
    outcome_results = []
    
    for model_name, pipe in [('Logistic Regression', pipe_lr), ('Gradient Boosting', pipe_gb)]:
        # Get predicted probabilities
        y_prob = cross_val_predict(pipe, X, y, cv=cv, method='predict_proba')[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        
        # Calculate metrics
        accuracy = accuracy_score(y, y_pred)
        auc = roc_auc_score(y, y_prob)
        brier = brier_score_loss(y, y_prob)
        
        result = {
            'outcome': outcome_name,
            'model': model_name,
            'accuracy': accuracy,
            'roc_auc': auc,
            'brier_score': brier,
            'support': y.sum()
        }
        outcome_results.append(result)
        print(f'  {model_name}: AUC={auc:.3f}  Acc={accuracy:.3f}  Brier={brier:.3f}')
    
    # Store best model (by AUC)
    best_result = max(outcome_results, key=lambda x: x['roc_auc'])
    best_models[outcome_name] = best_result['model']
    
    # Get probabilities from best model
    best_pipe = pipe_lr if best_result['model'] == 'Logistic Regression' else pipe_gb
    y_prob_best = cross_val_predict(best_pipe, X, y, cv=cv, method='predict_proba')[:, 1]
    all_probs[outcome_name] = y_prob_best
    
    all_results.extend(outcome_results)

# Create results dataframe
df_results = pd.DataFrame(all_results)
df_results.to_csv(REPORT_DIR / 'binary_model_results.csv', index=False)

print('\n=== Summary ===')
print(df_results.pivot(index='outcome', columns='model', 
                       values=['roc_auc', 'accuracy']).round(3))

## 10.6 — Calibration Curves & Feature Importance by Outcome

In [ ]:
## 10.6 — Model Comparison: OpenAI vs Alternative Approaches

Comparing multiple embedding methods and algorithms:
- Embeddings: OpenAI, TF-IDF+SVD, Sentence-BERT
- Algorithms: Logistic Regression, XGBoost, SVM, TabNet

# First, let's prepare text data for alternative embeddings
if not TEXT_DIR.exists():
    print('Warning: TEXT_DIR not found. Using OpenAI embeddings only.')
    text_documents = [''] * len(df_obs)
else:
    # Load text documents matching our observations
    text_docs = {}
    for txt_path in TEXT_DIR.glob('CASE_*.txt'):
        case_id = txt_path.stem
        text = txt_path.read_text(encoding='utf-8', errors='ignore')[:8000]
        text_docs[case_id] = text
    
    # Map to observations
    text_documents = []
    for idx in df_obs['obs_id']:
        # For now, use modulo as case mapping (should be improved with actual mapping)
        case_id = f'CASE_{(idx % len(text_docs) + 1):03d}'
        text_documents.append(text_docs.get(case_id, ''))
    
    print(f'Loaded {len(text_documents)} text documents')

print('\n=== Alternative Embedding Methods ===')

# 1. TF-IDF + SVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

print('Generating TF-IDF + SVD embeddings...')
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), 
                       stop_words='english', min_df=2)
X_tfidf = tfidf.fit_transform(text_documents)

svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_svd = svd.fit_transform(X_tfidf)
print(f'  TF-IDF shape: {X_tfidf.shape}')
print(f'  SVD reduced shape: {X_svd.shape}')
print(f'  Explained variance: {svd.explained_variance_ratio_.sum():.3f}')

# 2. Sentence-BERT (optional - requires installation)
try:
    from sentence_transformers import SentenceTransformer
    print('\nGenerating Sentence-BERT embeddings...')
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    X_sbert = sbert_model.encode(text_documents, show_progress_bar=True)
    print(f'  SBERT shape: {X_sbert.shape}')
    sbert_available = True
except ImportError:
    print('\nSentence-BERT not available (install: pip install sentence-transformers)')
    X_sbert = None
    sbert_available = False

# Combine with categorical features
X_cat = pd.get_dummies(df_obs[['element', 'tumor_type', 'complex']], drop_first=True).fillna(0).values

# Prepare embedding combinations
embedding_sets = {
    'OpenAI': (X_pca.values if len(df_pca) > 0 else X[:, -50:]),  # Last 50 cols assumed PCA
    'TFIDF+SVD': X_svd,
    'SBERT': X_sbert if sbert_available else None
}

# Remove None values
embedding_sets = {k: v for k, v in embedding_sets.items() if v is not None}

print(f'\nEmbedding sets available: {list(embedding_sets.keys())}')

# Now compare models across embeddings for all three outcomes
print('\n=== Model Comparison Across Embeddings ===')
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Define models to test
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.1, 
                                             class_weight='balanced',
                                             random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                                   learning_rate=0.05, 
                                                   random_state=RANDOM_STATE),
    'SVM (Linear)': CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', random_state=RANDOM_STATE), cv=3)
}

# Add TabNet if available
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    models['TabNet'] = TabNetClassifier(
        n_d=64, n_a=64, n_steps=5, gamma=1.5,
        lambda_sparse=1e-4, optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
        mask_type='entmax', scheduler_params=dict(step_size=50, gamma=0.9),
        verbose=0, seed=RANDOM_STATE
    )
    tabnet_available = True
    print('TabNet model added to comparison')
except ImportError:
    print('\nTabNet not available (install: pip install pytorch-tabnet)')
    tabnet_available = False

# Test on all three outcomes
outcomes = {
    'Correct': df_obs['is_correct'].values,
    'Omission': df_obs['is_omission'].values,
    'Fabrication': df_obs['is_fabrication'].values
}

all_results = []

for outcome_name, y_true in outcomes.items():
    print(f'\n=== Testing {outcome_name} Prediction ===')
    
    for embed_name, X_embed in embedding_sets.items():
        # Combine with categorical features
        X_combined = np.hstack([X_embed, X_cat])
        X_combined = StandardScaler().fit_transform(X_combined)
        
        print(f'\n--- {embed_name} Embeddings ---')
        
        for model_name, model in models.items():
            # Special handling for TabNet
            if model_name == 'TabNet' and tabnet_available:
                # TabNet requires different input format
                cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
                aucs = []
                
                for train_idx, val_idx in cv.split(X_combined, y_true):
                    X_train, X_val = X_combined[train_idx], X_combined[val_idx]
                    y_train, y_val = y_true[train_idx], y_true[val_idx]
                    
                    model.fit(X_train, y_train, 
                             eval_set=[(X_val, y_val)],
                             eval_metric=['auc'],
                             max_epochs=100, patience=10, batch_size=256, virtual_batch_size=128,
                             num_workers=0, drop_last=False)
                    
                    y_prob = model.predict_proba(X_val)[:, 1]
                    aucs.append(roc_auc_score(y_val, y_prob))
                
                auc = np.mean(aucs)
                accuracy = np.mean([accuracy_score(y_true[val_idx], 
                                                (model.predict_proba(X_combined[val_idx])[:, 1] >= 0.5).astype(int))
                                  for train_idx, val_idx in cv.split(X_combined, y_true)])
                f1 = np.mean([classification_report(y_true[val_idx], 
                                                  (model.predict_proba(X_combined[val_idx])[:, 1] >= 0.5).astype(int),
                                                  output_dict=True, zero_division=0)['1']['f1-score']
                             for train_idx, val_idx in cv.split(X_combined, y_true)])
            else:
                # Standard models
                cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
                y_prob = cross_val_predict(model, X_combined, y_true, 
                                         cv=cv, method='predict_proba')[:, 1]
                y_pred = (y_prob >= 0.5).astype(int)
                
                # Metrics
                auc = roc_auc_score(y_true, y_prob)
                accuracy = accuracy_score(y_true, y_pred)
                f1 = classification_report(y_true, y_pred, output_dict=True, 
                                          zero_division=0)['1']['f1-score']
            
            result = {
                'outcome': outcome_name,
                'embedding': embed_name,
                'model': model_name,
                'auc': auc,
                'accuracy': accuracy,
                'f1': f1
            }
            all_results.append(result)
            
            print(f'  {model_name}: AUC={auc:.3f}, F1={f1:.3f}')

# Create comprehensive comparison table
df_comparison = pd.DataFrame(all_results)
df_comparison.to_csv(REPORT_DIR / 'comprehensive_model_comparison.csv', index=False)

# Create summary tables for each outcome
for outcome in outcomes.keys():
    print(f'\n=== {outcome} Prediction Summary ===')
    summary = df_comparison[df_comparison['outcome'] == outcome].pivot(
        index='embedding', columns='model', values='auc'
    )
    print(summary.round(3))
    
    # Save individual outcome visualizations
    plt.figure(figsize=(12, 8))
    outcome_data = df_comparison[df_comparison['outcome'] == outcome]
    sns.barplot(data=outcome_data, x='embedding', y='auc', hue='model')
    plt.title(f'Model Comparison: AUC for {outcome} Prediction', fontsize=14)
    plt.ylabel('AUC Score')
    plt.xlabel('Embedding Method')
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f'model_comparison_{outcome.lower()}_auc.png', dpi=300)
    plt.show()
    
    # Heatmap for this outcome
    plt.figure(figsize=(10, 6))
    sns.heatmap(summary, annot=True, fmt='.3f', cmap='RdYlBu_r', center=0.5)
    plt.title(f'AUC Heatmap: Models vs Embeddings ({outcome})', fontsize=14)
    plt.tight_layout()
    plt.savefig(REPORT_DIR / f'model_comparison_{outcome.lower()}_heatmap.png', dpi=300)
    plt.show()

# Overall best performers
print('\n=== Best Performing Models by Outcome ===')
for outcome in outcomes.keys():
    best = df_comparison[df_comparison['outcome'] == outcome].nlargest(1, 'auc')
    print(f"{outcome}: {best['model'].iloc[0]} with {best['embedding'].iloc[0]} (AUC={best['auc'].iloc[0]:.3f})")

## 10.7 — Model D: Element-level Outcome Rate Prediction (Multivariate Regression)

Features: element identity, domain, n_cases  
Target: accuracy, omission_rate, fabrication_rate per element  
Model: Multi-output Ridge regression (leave-one-element-out CV)

In [ ]:
# Load prompt library if available
prompt_lib_path = PROJECT_ROOT / 'data' / 'processed' / 'prompt_library_updated_v5.xlsx'
if not prompt_lib_path.exists():
    prompt_lib_path = PROJECT_ROOT / 'data' / 'processed' / 'prompt_library_updated_v4.xlsx'
if not prompt_lib_path.exists():
    prompt_lib_path = PROJECT_ROOT / 'prompts' / 'prompt_library.csv'

df_prompts = None
if prompt_lib_path.exists():
    if prompt_lib_path.suffix in ['.xlsx', '.xls']:
        df_prompts = pd.read_excel(prompt_lib_path)
    else:
        df_prompts = pd.read_csv(prompt_lib_path)
    print(f'Prompt library loaded: {df_prompts.shape}')
    print(f'Columns: {list(df_prompts.columns[:10])}')
else:
    print('Prompt library not found — using element-level data only.')

# Domain classification
RAD_ELEMENTS  = {'Lesion Size', 'Lesion Laterality', 'Lesion Location',
                 'Cal_asym', 'MRI Enhancement', 'Extent',
                 'Clip Placement', 'Workup Recommendation', 'Lymph Node'}

df_run['domain'] = df_run['element'].apply(
    lambda e: 'Radiology' if any(r.lower() in e.lower() for r in
              ['lesion', 'lateral', 'location', 'calcif', 'mri', 'extent',
               'clip', 'workup', 'lymph'])
    else 'Pathology')

# Build X_B from element features
X_B_df = pd.get_dummies(df_run[['domain', 'n_cases']], drop_first=True)
X_B    = X_B_df.fillna(0).values
y_B_acc = df_run['ai_accuracy'].values
y_B_fab = df_run['fabrication_rate'].fillna(0).values

print(f'Model B design matrix: {X_B.shape}')
print(f'y_B accuracy range: {y_B_acc.min():.3f} – {y_B_acc.max():.3f}')

In [ ]:
# Leave-one-element-out multi-output ridge regression
from sklearn.multioutput import MultiOutputRegressor

ridge_multi = MultiOutputRegressor(Ridge(alpha=1.0), n_jobs=-1)

# Prepare multi-target: [accuracy, omission_rate, fabrication_rate]
y_B_multi = df_run[['accuracy', 'omission_rate', 'fabrication_rate']].values

loo = LeaveOneOut()
y_pred_multi = cross_val_predict(ridge_multi, X_B, y_B_multi, cv=loo)

# Calculate metrics for each outcome
outcomes = ['accuracy', 'omission_rate', 'fabrication_rate']
results_b = {}
for i, outcome in enumerate(outcomes):
    y_true = y_B_multi[:, i]
    y_pred = y_pred_multi[:, i]
    results_b[f'{outcome}_rmse'] = np.sqrt(mean_squared_error(y_true, y_pred))
    results_b[f'{outcome}_r2'] = r2_score(y_true, y_pred)

pd.DataFrame([results_b]).to_csv(REPORT_DIR / 'model_b_regression_results.csv', index=False)

print('Model B Results (Leave-One-Element-Out Multi-output Ridge Regression):')
for outcome in outcomes:
    print(f'  {outcome}: R²={results_b[f"{outcome}_r2"]:.3f}, RMSE={results_b[f"{outcome}_rmse"]:.4f}')

# ── Plot predicted vs actual for all three outcomes ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#27ae60', '#f39c12', '#e74c3c']

for ax, outcome, color in zip(axes, outcomes, colors):
    y_true = y_B_multi[:, outcomes.index(outcome)]
    y_pred = y_pred_multi[:, outcomes.index(outcome)]
    
    ax.scatter(y_true, y_pred, color=color, edgecolor='black',
               s=80, alpha=0.8, linewidth=0.5)
    lim = [min(y_true.min(), y_pred.min()) - 0.02,
           max(y_true.max(), y_pred.max()) + 0.02]
    ax.plot(lim, lim, '--', color='navy', linewidth=1.2, label='Perfect fit')
    
    # Label each point with element name
    for i, elem in enumerate(df_run['element']):
        ax.annotate(elem.replace('_', ' ')[:15], (y_true[i], y_pred[i]),
                    fontsize=6.5, alpha=0.75, xytext=(3, 3),
                    textcoords='offset points')
    
    ax.set_xlabel(f'Actual {outcome}')
    ax.set_ylabel(f'Predicted {outcome}')
    ax.set_title(f'{outcome.title()}\nR²={results_b[f"{outcome}_r2"]:.3f}  '
                 f'RMSE={results_b[f"{outcome}_rmse"]:.4f}',
                 fontweight='bold', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Model B: Multi-output Prediction by Element',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'model_b_multivariate_regression.png', dpi=200, bbox_inches='tight')
plt.show()

## 10.8 — Summary: Combined Feature Importance Dashboard

In [ ]:
# Summary table: All model performance
summary_rows = []

# Add binary model results
for _, row in df_results.iterrows():
    summary_rows.append({
        'outcome_type': 'Observation-level',
        'outcome': row['outcome'],
        'model': row['model'],
        'metric': 'AUC',
        'value': row['roc_auc']
    })
    summary_rows.append({
        'outcome_type': 'Observation-level',
        'outcome': row['outcome'],
        'model': row['model'],
        'metric': 'Accuracy',
        'value': row['accuracy']
    })

# Add regression results (from cell 17)
for outcome in ['accuracy', 'omission_rate', 'fabrication_rate']:
    summary_rows.append({
        'outcome_type': 'Element-level',
        'outcome': outcome.replace('_', ' ').title(),
        'model': 'Ridge LOO',
        'metric': 'R²',
        'value': results_b[f'{outcome}_r2']
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(REPORT_DIR / 'openai_model_summary.csv', index=False)

print('=' * 70)
print('PREDICTIVE MODEL SUMMARY - BINARY OUTCOMES')
print('=' * 70)
# Format for better display
for outcome in ['Correct', 'Omission', 'Fabrication']:
    print(f'\n{outcome}:')
    subset = df_results[df_results['outcome'] == outcome]
    for _, row in subset.iterrows():
        print(f'  {row["model"]}: AUC={row["roc_auc"]:.3f}, Acc={row["accuracy"]:.3f}')

print(f'\nElement-level predictions (R²):')
for outcome in ['accuracy', 'omission_rate', 'fabrication_rate']:
    print(f'  {outcome}: {results_b[f"{outcome}_r2"]:.3f}')

print('=' * 70)
print('\nKey Insights:')
print('  • Binary models allow independent predictors for each outcome type')
print('  • Fabrication model uses class weighting to handle rarity')
print('  • Different features may drive correct vs omission vs fabrication')
print('  • Element-level regression reveals systematic patterns')
print(f'\nAll outputs saved to: {REPORT_DIR}')